# CRM Access Governance & Customer Data Protection
## Stage 3 of 12 — Governance Rules

### Project Position

This notebook is the third stage of the CRM Access Governance & Customer Data Protection roadmap.

- **Stage 1 — EDA:** identified patterns, distributions, and candidate risk signals.
- **Stage 2 — Explanatory Analysis:** measured associations and tested which factors appear to explain `Access_Decision`.
- **Stage 3 — Governance Rules:** translates that evidence into proposed, transparent, auditable, and reviewable governance rules.

### Core distinction

This notebook explicitly separates three concepts:

1. **Observed Evidence** — patterns supported by Stages 1 and 2.
2. **Proposed Governance Policy** — rules designed from that evidence and business-governance reasoning.
3. **Implemented Control** — a future production control, which would require business, security, privacy, legal, and system-owner approval.

> **Important:** this is a governance prototype for a synthetic CRM environment. It is not legal advice, not a production security policy, and not a direct statement of LGPD/GDPR requirements.


In [1]:
# 1. Libraries and Settings

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 300)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

DATA_PATH = Path("Permission_Aware_CRM_Governance_Synthetic_50000.csv")

print("Environment ready.")


Environment ready.


# 2. Data Loading

The original synthetic dataset is loaded again so the proposed governance framework can be tested against the observed access records.

`Is_Blocked` is created only as a validation field for comparison with the historical synthetic outcome.


In [2]:
df = pd.read_csv(DATA_PATH)

df_rules = df.copy()

df_rules["Is_Blocked"] = (
    df_rules["Access_Decision"]
    .eq("Block")
    .astype(int)
)

print(f"Rows: {df_rules.shape[0]:,}")
print(f"Columns: {df_rules.shape[1]}")
display(df_rules.head())


Rows: 50,000
Columns: 16


,User_ID,Role,Region,Lead_Source,CRM_Action,Daily_Logins,Failed_Logins,Access_Hour,Device_Type,Data_Sensitivity,Policy_Compliance_Score,Anomaly_Score,Permission_Granted,Governance_Score,Access_Decision,Is_Blocked
0,1,Admin,North,Referral,ViewLead,6,1,10,Managed,5,76.190,0.114,True,51.810,Block,1
1,2,Sales Rep,West,Web,ViewLead,5,1,12,Managed,5,62.870,0.464,True,38.400,Block,1
2,3,Sales Rep,South,Partner,EditLead,6,1,20,Managed,1,81.870,0.083,False,48.570,Block,1
3,4,Analyst,South,Campaign,ViewLead,10,0,10,Managed,3,85.890,0.181,True,57.530,Block,1
4,5,Sales Rep,West,Web,EditLead,6,2,17,Managed,1,92.310,0.309,False,44.580,Block,1


# 3. Inputs from Stages 1 and 2

Stage 3 does not start from a blank policy. It uses analytical evidence generated previously.

### Evidence carried forward

- `Permission_Granted = False` is strongly associated with blocking.
- Higher `Data_Sensitivity` is associated with higher block rates.
- BYOD access shows a higher observed block rate than managed-device access.
- More failed login attempts are associated with higher block rates.
- Higher `Anomaly_Score` is associated with blocking.
- Higher `Policy_Compliance_Score` is associated with more favorable outcomes.
- Multiple contextual risk factors together are more informative than isolated factors.
- `Governance_Score` strongly separates the target classes, but Stage 2 identified a **proxy/leakage-like risk**.

### Stage 3 consequence

`Governance_Score` will **not** be used to define the operational contextual risk score or the proposed decision engine. It may still be retained later as a diagnostic or monitoring indicator.


# 4. Governance Design Principles

The framework follows a layered decision model.

### Layer 1 — Authorization
Does the user have explicit permission, and is the requested action consistent with the proposed role baseline?

### Layer 2 — Contextual Risk
Even when authorization exists, contextual signals may increase risk:

- higher data sensitivity;
- BYOD / unmanaged-device context;
- failed login attempts;
- anomalous behavior;
- unusual access time;
- low policy compliance.

### Layer 3 — Decision
The framework produces one of three proposed governance states:

- `ALLOW`
- `REVIEW`
- `BLOCK`

### Layer 4 — Human Governance
Rules that produce `REVIEW`, exceptions, and policy disagreements require human validation rather than automatic policy expansion.

The goal is **least privilege + contextual review + traceability**, not maximum agreement with the synthetic target.


# 5. Proposed Internal Data Sensitivity Classification

The original dataset contains `Data_Sensitivity` levels from 1 to 5.

For Stage 3, those levels are mapped only to **internal governance sensitivity labels**.

They do **not** indicate whether a field is personal data, sensitive personal data, or another legal privacy category. Formal PII/LGPD classification belongs to **Stage 6 — Data Classification & Privacy Engineering**.


In [3]:
sensitivity_policy = pd.DataFrame({
    "Data_Sensitivity": [1, 2, 3, 4, 5],
    "Sensitivity_Label": [
        "Low",
        "Internal",
        "Confidential",
        "Restricted",
        "Highly Restricted"
    ],
    "Default_Control": [
        "Standard access",
        "Standard access with logging",
        "Role-based restriction",
        "Enhanced contextual review",
        "Strict restriction and enhanced review"
    ],
    "Policy_Status": [
        "Proposed",
        "Proposed",
        "Proposed",
        "Proposed",
        "Proposed"
    ]
})

display(sensitivity_policy)


,Data_Sensitivity,Sensitivity_Label,Default_Control,Policy_Status
0,1,Low,Standard access,Proposed
1,2,Internal,Standard access with logging,Proposed
2,3,Confidential,Role-based restriction,Proposed
3,4,Restricted,Enhanced contextual review,Proposed
4,5,Highly Restricted,Strict restriction and enhanced review,Proposed


## Governance Note

These labels are internal project conventions created to support access-governance design.

They must remain separate from:

- PII classification;
- LGPD legal categories;
- retention requirements;
- lawful-basis assessment;
- privacy rights.

Those topics are handled later in the roadmap.


# 6. Proposed Role × CRM Action Access Matrix

This matrix defines a **proposed RBAC baseline** for the simulated CRM.

Allowed values:

- `ALLOW` — the role is normally permitted to perform the action.
- `REVIEW` — the action may be legitimate but requires additional approval or contextual review.
- `BLOCK` — the action is outside the proposed baseline for the role.

This matrix is a **policy design artifact**, not a pattern inferred directly from the dataset.


In [4]:
roles = [
    "Admin",
    "Manager",
    "Sales Rep",
    "Support",
    "Analyst"
]

actions = [
    "ViewLead",
    "EditLead",
    "CreateOpportunity",
    "ApproveDiscount",
    "ExportCRM",
    "DeleteLead"
]

access_matrix = pd.DataFrame(
    index=roles,
    columns=actions
)

access_matrix.loc["Admin"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "ALLOW", "ALLOW", "ALLOW"
]

access_matrix.loc["Manager"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "ALLOW", "REVIEW", "REVIEW"
]

access_matrix.loc["Sales Rep"] = [
    "ALLOW", "ALLOW", "ALLOW",
    "REVIEW", "REVIEW", "BLOCK"
]

access_matrix.loc["Support"] = [
    "ALLOW", "REVIEW", "BLOCK",
    "BLOCK", "BLOCK", "BLOCK"
]

access_matrix.loc["Analyst"] = [
    "ALLOW", "BLOCK", "BLOCK",
    "BLOCK", "REVIEW", "BLOCK"
]

display(access_matrix)


,ViewLead,EditLead,CreateOpportunity,ApproveDiscount,ExportCRM,DeleteLead
Admin,ALLOW,ALLOW,ALLOW,ALLOW,ALLOW,ALLOW
Manager,ALLOW,ALLOW,ALLOW,ALLOW,REVIEW,REVIEW
Sales Rep,ALLOW,ALLOW,ALLOW,REVIEW,REVIEW,BLOCK
Support,ALLOW,REVIEW,BLOCK,BLOCK,BLOCK,BLOCK
Analyst,ALLOW,BLOCK,BLOCK,BLOCK,REVIEW,BLOCK


## Interpretation

The matrix should be read as a proposed least-privilege baseline.

A real organization would require validation by business owners, security, IAM, privacy/compliance, and system administrators before any row-action combination became an implemented control.


# 7. Convert Access Matrix to Long Format

The long format makes the RBAC matrix reusable for joins, documentation, exports, monitoring, and later Power BI modeling.


In [5]:
access_matrix_long = (
    access_matrix
    .reset_index()
    .rename(columns={"index": "Role"})
    .melt(
        id_vars="Role",
        var_name="CRM_Action",
        value_name="Baseline_Authorization"
    )
)

display(access_matrix_long.head(20))


,Role,CRM_Action,Baseline_Authorization
0,Admin,ViewLead,ALLOW
1,Manager,ViewLead,ALLOW
2,Sales Rep,ViewLead,ALLOW
3,Support,ViewLead,ALLOW
4,Analyst,ViewLead,ALLOW
5,Admin,EditLead,ALLOW
6,Manager,EditLead,ALLOW
7,Sales Rep,EditLead,ALLOW
8,Support,EditLead,REVIEW
9,Analyst,EditLead,BLOCK


# 8. Contextual Risk Flags

Contextual controls operate **after authorization has been evaluated**.

The flags below are intentionally transparent and interpretable.

### Important analytical boundary

`Governance_Score` is excluded from the operational risk flags because Stage 2 showed that it may encode a large part of the synthetic decision boundary. Using it here would risk turning a derived score into a circular policy input.


In [6]:
df_rules["High_Sensitivity_Flag"] = (
    df_rules["Data_Sensitivity"] >= 4
)

df_rules["BYOD_Flag"] = (
    df_rules["Device_Type"] == "BYOD"
)

df_rules["Failed_Login_Flag"] = (
    df_rules["Failed_Logins"] > 0
)

# Data-driven threshold used only as a proposed exploratory boundary.
anomaly_threshold = df_rules["Anomaly_Score"].quantile(0.90)

df_rules["High_Anomaly_Flag"] = (
    df_rules["Anomaly_Score"] >= anomaly_threshold
)

min_hour = df_rules["Access_Hour"].min()
max_hour = df_rules["Access_Hour"].max()

df_rules["Edge_Hour_Flag"] = (
    (df_rules["Access_Hour"] <= min_hour + 1) |
    (df_rules["Access_Hour"] >= max_hour - 1)
)

compliance_threshold = df_rules["Policy_Compliance_Score"].quantile(0.25)

df_rules["Low_Compliance_Flag"] = (
    df_rules["Policy_Compliance_Score"] < compliance_threshold
)

print(f"High anomaly threshold (90th percentile): {anomaly_threshold:.3f}")
print(f"Low compliance threshold (25th percentile): {compliance_threshold:.3f}")


High anomaly threshold (90th percentile): 0.389
Low compliance threshold (25th percentile): 75.170


# 9. Contextual Risk Score

A transparent count-based score is used as a **governance prototype**.

Each triggered flag contributes one point. Equal weighting is intentional for interpretability, not because the factors are proven to have equal real-world importance.

The score therefore represents:

> **number of concurrent contextual risk signals**

It is not a probability of fraud, a legal-risk score, or a production security score.


In [7]:
contextual_flags = [
    "High_Sensitivity_Flag",
    "BYOD_Flag",
    "Failed_Login_Flag",
    "High_Anomaly_Flag",
    "Edge_Hour_Flag",
    "Low_Compliance_Flag"
]

df_rules["Contextual_Risk_Score"] = (
    df_rules[contextual_flags]
    .sum(axis=1)
)

risk_level_map = {
    0: "LOW",
    1: "LOW",
    2: "MEDIUM",
    3: "MEDIUM",
    4: "HIGH",
    5: "HIGH",
    6: "CRITICAL"
}

df_rules["Contextual_Risk_Level"] = (
    df_rules["Contextual_Risk_Score"]
    .map(risk_level_map)
)

risk_distribution = (
    df_rules["Contextual_Risk_Level"]
    .value_counts()
    .reindex(["LOW", "MEDIUM", "HIGH", "CRITICAL"], fill_value=0)
    .to_frame("Records")
)

risk_distribution["Percentage"] = (
    risk_distribution["Records"] / len(df_rules) * 100
).round(2)

display(
    df_rules[
        contextual_flags +
        ["Contextual_Risk_Score", "Contextual_Risk_Level"]
    ].head()
)

display(risk_distribution)


,High_Sensitivity_Flag,BYOD_Flag,Failed_Login_Flag,High_Anomaly_Flag,Edge_Hour_Flag,Low_Compliance_Flag,Contextual_Risk_Score,Contextual_Risk_Level
0,True,False,True,False,False,False,2,MEDIUM
1,True,False,True,True,False,True,4,HIGH
2,False,False,True,False,False,False,1,LOW
3,False,False,False,False,False,False,0,LOW
4,False,False,True,False,False,False,1,LOW


,Records,Percentage
Contextual_Risk_Level,,
LOW,24919,49.840
MEDIUM,22974,45.950
HIGH,2087,4.170
CRITICAL,20,0.040


# 10. Evidence-to-Policy Boundary

Before defining rules, the notebook records whether each element is:

- **Observed Evidence** — directly supported by prior analysis;
- **Proposed Policy** — a governance decision informed by evidence;
- **Future Control** — something that would only exist after organizational approval and implementation.

This prevents statistical findings from being silently converted into mandatory policy.


In [8]:
evidence_policy_map = pd.DataFrame([
    {
        "Topic": "Explicit permission",
        "Observed_Evidence": "Permission_Granted=False is strongly associated with blocking.",
        "Proposed_Policy": "No explicit permission should result in BLOCK.",
        "Policy_Status": "Proposed",
        "Future_Control": "IAM / access-control enforcement"
    },
    {
        "Topic": "Role × CRM Action",
        "Observed_Evidence": "Roles and actions show different block patterns.",
        "Proposed_Policy": "Use a least-privilege RBAC baseline by role and action.",
        "Policy_Status": "Proposed",
        "Future_Control": "RBAC matrix in CRM/IAM"
    },
    {
        "Topic": "Data sensitivity",
        "Observed_Evidence": "Higher sensitivity is associated with higher block rates.",
        "Proposed_Policy": "Sensitivity levels 4–5 require enhanced contextual review.",
        "Policy_Status": "Proposed",
        "Future_Control": "Sensitivity-aware access policy"
    },
    {
        "Topic": "BYOD",
        "Observed_Evidence": "BYOD shows a higher observed block rate than managed devices.",
        "Proposed_Policy": "BYOD contributes one contextual risk signal.",
        "Policy_Status": "Proposed",
        "Future_Control": "Device posture / conditional access"
    },
    {
        "Topic": "Failed logins",
        "Observed_Evidence": "Block rate increases with failed-login count.",
        "Proposed_Policy": "Any failed login contributes one contextual risk signal.",
        "Policy_Status": "Proposed",
        "Future_Control": "Authentication-risk monitoring"
    },
    {
        "Topic": "Anomaly score",
        "Observed_Evidence": "Higher anomaly scores are associated with blocking.",
        "Proposed_Policy": "Top-decile anomaly is treated as a high-anomaly flag for this prototype.",
        "Policy_Status": "Proposed threshold",
        "Future_Control": "Behavioral anomaly monitoring"
    },
    {
        "Topic": "Governance score",
        "Observed_Evidence": "Strongly separates synthetic decision classes.",
        "Proposed_Policy": "Exclude from decision-engine inputs due to proxy/leakage-like risk.",
        "Policy_Status": "Analytical boundary",
        "Future_Control": "Diagnostic monitoring only"
    }
])

display(evidence_policy_map)


,Topic,Observed_Evidence,Proposed_Policy,Policy_Status,Future_Control
0,Explicit permission,Permission_Granted=False is strongly associate...,No explicit permission should result in BLOCK.,Proposed,IAM / access-control enforcement
1,Role × CRM Action,Roles and actions show different block patterns.,Use a least-privilege RBAC baseline by role an...,Proposed,RBAC matrix in CRM/IAM
2,Data sensitivity,Higher sensitivity is associated with higher b...,Sensitivity levels 4–5 require enhanced contex...,Proposed,Sensitivity-aware access policy
3,BYOD,BYOD shows a higher observed block rate than m...,BYOD contributes one contextual risk signal.,Proposed,Device posture / conditional access
4,Failed logins,Block rate increases with failed-login count.,Any failed login contributes one contextual ri...,Proposed,Authentication-risk monitoring
5,Anomaly score,Higher anomaly scores are associated with bloc...,Top-decile anomaly is treated as a high-anomal...,Proposed threshold,Behavioral anomaly monitoring
6,Governance score,Strongly separates synthetic decision classes.,Exclude from decision-engine inputs due to pro...,Analytical boundary,Diagnostic monitoring only


# 11. Rule Catalog

Each proposed governance rule receives a stable identifier and traceability metadata.

The catalog separates:

- rule category;
- priority;
- trigger condition;
- proposed outcome;
- rationale;
- evidence source;
- policy status;
- future control type.

This makes the framework auditable and easier to maintain.


In [9]:
rule_catalog = pd.DataFrame([
    {
        "Rule_ID": "AUTH-001",
        "Category": "Authorization",
        "Priority": 1,
        "Condition": "Permission_Granted = False",
        "Proposed_Decision": "BLOCK",
        "Rationale": "No explicit permission is available for the requested access.",
        "Evidence_Source": "Stage 2 — permission analysis",
        "Policy_Status": "Proposed",
        "Control_Type": "Preventive"
    },
    {
        "Rule_ID": "AUTH-002",
        "Category": "Authorization",
        "Priority": 2,
        "Condition": "Role × CRM_Action baseline = BLOCK",
        "Proposed_Decision": "BLOCK",
        "Rationale": "Requested action is outside the proposed role authorization baseline.",
        "Evidence_Source": "Stage 3 — RBAC governance design",
        "Policy_Status": "Proposed",
        "Control_Type": "Preventive"
    },
    {
        "Rule_ID": "CTX-001",
        "Category": "Contextual Risk",
        "Priority": 3,
        "Condition": "Contextual_Risk_Level = CRITICAL",
        "Proposed_Decision": "BLOCK",
        "Rationale": "Multiple concurrent contextual risk signals indicate a critical access context.",
        "Evidence_Source": "Stages 1–2 — combined risk-factor evidence",
        "Policy_Status": "Proposed",
        "Control_Type": "Preventive"
    },
    {
        "Rule_ID": "CTX-002",
        "Category": "Contextual Risk",
        "Priority": 4,
        "Condition": "High sensitivity AND BYOD AND high anomaly",
        "Proposed_Decision": "BLOCK",
        "Rationale": "Highly sensitive access from BYOD with anomalous behavior requires strict treatment.",
        "Evidence_Source": "Stage 2 — interaction evidence + governance design",
        "Policy_Status": "Proposed",
        "Control_Type": "Preventive"
    },
    {
        "Rule_ID": "AUTH-003",
        "Category": "Authorization",
        "Priority": 5,
        "Condition": "Role × CRM_Action baseline = REVIEW",
        "Proposed_Decision": "REVIEW",
        "Rationale": "Action requires additional approval or contextual review.",
        "Evidence_Source": "Stage 3 — RBAC governance design",
        "Policy_Status": "Proposed",
        "Control_Type": "Detective / Approval"
    },
    {
        "Rule_ID": "CTX-003",
        "Category": "Contextual Risk",
        "Priority": 6,
        "Condition": "Contextual_Risk_Level = HIGH",
        "Proposed_Decision": "REVIEW",
        "Rationale": "High contextual risk requires manual or secondary validation.",
        "Evidence_Source": "Stages 1–2 — combined risk-factor evidence",
        "Policy_Status": "Proposed",
        "Control_Type": "Detective / Approval"
    },
    {
        "Rule_ID": "CTX-004",
        "Category": "Contextual Risk",
        "Priority": 7,
        "Condition": "Data_Sensitivity >= 4 AND Failed_Logins > 0",
        "Proposed_Decision": "REVIEW",
        "Rationale": "Sensitive-data access combined with authentication friction requires review.",
        "Evidence_Source": "Stages 1–2 — sensitivity and failed-login evidence",
        "Policy_Status": "Proposed",
        "Control_Type": "Detective / Approval"
    },
    {
        "Rule_ID": "CTX-005",
        "Category": "Contextual Risk",
        "Priority": 8,
        "Condition": "Contextual_Risk_Level = MEDIUM",
        "Proposed_Decision": "REVIEW",
        "Rationale": "Moderate contextual risk requires additional attention.",
        "Evidence_Source": "Stages 1–2 — combined risk-factor evidence",
        "Policy_Status": "Proposed",
        "Control_Type": "Detective / Approval"
    },
    {
        "Rule_ID": "DEFAULT-001",
        "Category": "Default",
        "Priority": 99,
        "Condition": "No higher-priority rule triggered",
        "Proposed_Decision": "ALLOW",
        "Rationale": "Authorized low-risk access follows the default allow path.",
        "Evidence_Source": "Stage 3 — governance design",
        "Policy_Status": "Proposed",
        "Control_Type": "Default path"
    }
]).sort_values("Priority")

display(rule_catalog)


,Rule_ID,Category,Priority,Condition,Proposed_Decision,Rationale,Evidence_Source,Policy_Status,Control_Type
0,AUTH-001,Authorization,1,Permission_Granted = False,BLOCK,No explicit permission is available for the re...,Stage 2 — permission analysis,Proposed,Preventive
1,AUTH-002,Authorization,2,Role × CRM_Action baseline = BLOCK,BLOCK,Requested action is outside the proposed role ...,Stage 3 — RBAC governance design,Proposed,Preventive
2,CTX-001,Contextual Risk,3,Contextual_Risk_Level = CRITICAL,BLOCK,Multiple concurrent contextual risk signals in...,Stages 1–2 — combined risk-factor evidence,Proposed,Preventive
3,CTX-002,Contextual Risk,4,High sensitivity AND BYOD AND high anomaly,BLOCK,Highly sensitive access from BYOD with anomalo...,Stage 2 — interaction evidence + governance de...,Proposed,Preventive
4,AUTH-003,Authorization,5,Role × CRM_Action baseline = REVIEW,REVIEW,Action requires additional approval or context...,Stage 3 — RBAC governance design,Proposed,Detective / Approval
5,CTX-003,Contextual Risk,6,Contextual_Risk_Level = HIGH,REVIEW,High contextual risk requires manual or second...,Stages 1–2 — combined risk-factor evidence,Proposed,Detective / Approval
6,CTX-004,Contextual Risk,7,Data_Sensitivity >= 4 AND Failed_Logins > 0,REVIEW,Sensitive-data access combined with authentica...,Stages 1–2 — sensitivity and failed-login evid...,Proposed,Detective / Approval
7,CTX-005,Contextual Risk,8,Contextual_Risk_Level = MEDIUM,REVIEW,Moderate contextual risk requires additional a...,Stages 1–2 — combined risk-factor evidence,Proposed,Detective / Approval
8,DEFAULT-001,Default,99,No higher-priority rule triggered,ALLOW,Authorized low-risk access follows the default...,Stage 3 — governance design,Proposed,Default path


# 12. Join Baseline Authorization to Access Records

Each access record receives the proposed role-action baseline from the RBAC matrix.

Any unmapped combination must be treated as a governance-quality issue rather than silently defaulted.


In [10]:
df_rules = df_rules.merge(
    access_matrix_long,
    on=["Role", "CRM_Action"],
    how="left"
)

print(
    "Records without baseline authorization mapping:",
    df_rules["Baseline_Authorization"].isna().sum()
)

display(
    df_rules[
        [
            "Role",
            "CRM_Action",
            "Baseline_Authorization"
        ]
    ].head(10)
)


Records without baseline authorization mapping: 0


,Role,CRM_Action,Baseline_Authorization
0,Admin,ViewLead,ALLOW
1,Sales Rep,ViewLead,ALLOW
2,Sales Rep,EditLead,ALLOW
3,Analyst,ViewLead,ALLOW
4,Sales Rep,EditLead,ALLOW
5,Analyst,ViewLead,ALLOW
6,Support,ViewLead,ALLOW
7,Analyst,CreateOpportunity,BLOCK
8,Sales Rep,ExportCRM,REVIEW
9,Manager,EditLead,ALLOW


# 13. Governance Decision Engine

The rules are applied in a deterministic priority order.

### Proposed precedence

1. No explicit permission → `BLOCK`
2. Role-action baseline = `BLOCK` → `BLOCK`
3. Critical contextual risk → `BLOCK`
4. High sensitivity + BYOD + high anomaly → `BLOCK`
5. Role-action baseline = `REVIEW` → `REVIEW`
6. High contextual risk → `REVIEW`
7. High sensitivity + failed login → `REVIEW`
8. Medium contextual risk → `REVIEW`
9. Otherwise → `ALLOW`

The ordering is a **policy design choice**, not a statistical ranking.


In [11]:
def governance_decision(row):
    # Rule 1 — explicit permission
    if row["Permission_Granted"] == False:
        return "BLOCK", "AUTH-001"

    # Rule 2 — role-action baseline
    if row["Baseline_Authorization"] == "BLOCK":
        return "BLOCK", "AUTH-002"

    # Rule 3 — critical contextual risk
    if row["Contextual_Risk_Level"] == "CRITICAL":
        return "BLOCK", "CTX-001"

    # Rule 4 — highly sensitive anomalous BYOD access
    if (
        row["High_Sensitivity_Flag"]
        and row["BYOD_Flag"]
        and row["High_Anomaly_Flag"]
    ):
        return "BLOCK", "CTX-002"

    # Rule 5 — baseline requires review
    if row["Baseline_Authorization"] == "REVIEW":
        return "REVIEW", "AUTH-003"

    # Rule 6 — high contextual risk
    if row["Contextual_Risk_Level"] == "HIGH":
        return "REVIEW", "CTX-003"

    # Rule 7 — sensitive data with failed authentication attempts
    if (
        row["High_Sensitivity_Flag"]
        and row["Failed_Login_Flag"]
    ):
        return "REVIEW", "CTX-004"

    # Rule 8 — medium contextual risk
    if row["Contextual_Risk_Level"] == "MEDIUM":
        return "REVIEW", "CTX-005"

    # Default rule
    return "ALLOW", "DEFAULT-001"


In [12]:
decision_output = df_rules.apply(
    governance_decision,
    axis=1,
    result_type="expand"
)

decision_output.columns = [
    "Proposed_Access_Decision",
    "Triggered_Rule_ID"
]

df_rules = pd.concat(
    [df_rules, decision_output],
    axis=1
)

display(
    df_rules[
        [
            "User_ID",
            "Role",
            "CRM_Action",
            "Permission_Granted",
            "Baseline_Authorization",
            "Contextual_Risk_Level",
            "Proposed_Access_Decision",
            "Triggered_Rule_ID"
        ]
    ].head(20)
)


,User_ID,Role,CRM_Action,Permission_Granted,Baseline_Authorization,Contextual_Risk_Level,Proposed_Access_Decision,Triggered_Rule_ID
0,1,Admin,ViewLead,True,ALLOW,MEDIUM,REVIEW,CTX-004
1,2,Sales Rep,ViewLead,True,ALLOW,HIGH,REVIEW,CTX-003
2,3,Sales Rep,EditLead,False,ALLOW,LOW,BLOCK,AUTH-001
3,4,Analyst,ViewLead,True,ALLOW,LOW,ALLOW,DEFAULT-001
4,5,Sales Rep,EditLead,False,ALLOW,LOW,BLOCK,AUTH-001
5,6,Analyst,ViewLead,True,ALLOW,MEDIUM,REVIEW,CTX-004
6,7,Support,ViewLead,True,ALLOW,LOW,ALLOW,DEFAULT-001
7,8,Analyst,CreateOpportunity,True,BLOCK,LOW,BLOCK,AUTH-002
8,9,Sales Rep,ExportCRM,False,REVIEW,MEDIUM,BLOCK,AUTH-001
9,10,Manager,EditLead,True,ALLOW,MEDIUM,REVIEW,CTX-005


# 14. Proposed Decision Distribution

Unlike the original synthetic target, the proposed governance framework contains three outcomes: `ALLOW`, `REVIEW`, and `BLOCK`.

`ALLOW` is therefore a **proposed governance state**, not an outcome learned from the original target.


In [13]:
proposed_distribution = (
    df_rules["Proposed_Access_Decision"]
    .value_counts()
    .to_frame("Count")
    .assign(
        Percentage=lambda x:
        x["Count"] / len(df_rules) * 100
    )
)

display(proposed_distribution.round(2))


,Count,Percentage
Proposed_Access_Decision,,
BLOCK,19343,38.690
REVIEW,15587,31.170
ALLOW,15070,30.140


# 15. Rule Trigger Frequency

This table shows which proposed rules dominate the framework.

A rule that triggers extremely often may deserve review because it can overshadow lower-priority controls.


In [14]:
rule_trigger_summary = (
    df_rules["Triggered_Rule_ID"]
    .value_counts()
    .to_frame("Records")
    .reset_index()
    .rename(columns={"index": "Triggered_Rule_ID"})
)

rule_trigger_summary["Percentage"] = (
    rule_trigger_summary["Records"] /
    len(df_rules) * 100
)

rule_trigger_summary = rule_trigger_summary.merge(
    rule_catalog[
        [
            "Rule_ID",
            "Category",
            "Proposed_Decision",
            "Rationale"
        ]
    ],
    left_on="Triggered_Rule_ID",
    right_on="Rule_ID",
    how="left"
)

display(rule_trigger_summary.round(2))


,Triggered_Rule_ID,Records,Percentage,Rule_ID,Category,Proposed_Decision,Rationale
0,AUTH-001,15931,31.860,AUTH-001,Authorization,BLOCK,No explicit permission is available for the re...
1,DEFAULT-001,15070,30.140,DEFAULT-001,Default,ALLOW,Authorized low-risk access follows the default...
2,CTX-005,9745,19.490,CTX-005,Contextual Risk,REVIEW,Moderate contextual risk requires additional a...
3,CTX-004,4044,8.090,CTX-004,Contextual Risk,REVIEW,Sensitive-data access combined with authentica...
4,AUTH-002,2968,5.940,AUTH-002,Authorization,BLOCK,Requested action is outside the proposed role ...
5,CTX-003,986,1.970,CTX-003,Contextual Risk,REVIEW,High contextual risk requires manual or second...
6,AUTH-003,812,1.620,AUTH-003,Authorization,REVIEW,Action requires additional approval or context...
7,CTX-002,428,0.860,CTX-002,Contextual Risk,BLOCK,Highly sensitive access from BYOD with anomalo...
8,CTX-001,16,0.030,CTX-001,Contextual Risk,BLOCK,Multiple concurrent contextual risk signals in...


# 16. Comparison with the Synthetic Access Decision

This comparison is a **validation diagnostic**, not an optimization target.

The original dataset contains only `Block` and `Review`, while the proposed framework contains `ALLOW`, `REVIEW`, and `BLOCK`.

Therefore, disagreement does not automatically mean the proposed rule is wrong.


In [15]:
comparison = pd.crosstab(
    df_rules["Access_Decision"],
    df_rules["Proposed_Access_Decision"],
    margins=True
)

display(comparison)


Proposed_Access_Decision,ALLOW,BLOCK,REVIEW,All
Access_Decision,,,,
Block,9403,18647,14022,42072
Review,5667,696,1565,7928
All,15070,19343,15587,50000


# 17. Block-Decision Agreement

Because both systems contain a block outcome, block agreement can be examined separately.

This is useful for checking whether proposed preventive controls are broadly aligned with the synthetic historical pattern.


In [16]:
df_rules["Original_Block"] = (
    df_rules["Access_Decision"] == "Block"
)

df_rules["Proposed_Block"] = (
    df_rules["Proposed_Access_Decision"] == "BLOCK"
)

block_comparison = pd.crosstab(
    df_rules["Original_Block"],
    df_rules["Proposed_Block"],
    margins=True
)

display(block_comparison)


Proposed_Block,False,True,All
Original_Block,,,
False,7232,696,7928
True,23425,18647,42072
All,30657,19343,50000


# 18. Exceptions and Human Review Candidates

Records where the proposed framework and the synthetic decision differ are **validation candidates**.

They should not be automatically treated as policy errors.

Exception review should ask:

- Which rule triggered?
- Was the role-action baseline appropriate?
- Was the contextual threshold too strict or too permissive?
- Does the case require a formal exception process?


In [17]:
exceptions = df_rules[
    (
        (df_rules["Access_Decision"] == "Block") &
        (df_rules["Proposed_Access_Decision"] != "BLOCK")
    )
    |
    (
        (df_rules["Access_Decision"] == "Review") &
        (df_rules["Proposed_Access_Decision"] == "BLOCK")
    )
].copy()

print(f"Potential governance exceptions: {len(exceptions):,}")

display(
    exceptions[
        [
            "User_ID",
            "Role",
            "CRM_Action",
            "Device_Type",
            "Data_Sensitivity",
            "Failed_Logins",
            "Anomaly_Score",
            "Policy_Compliance_Score",
            "Governance_Score",
            "Permission_Granted",
            "Baseline_Authorization",
            "Contextual_Risk_Level",
            "Access_Decision",
            "Proposed_Access_Decision",
            "Triggered_Rule_ID"
        ]
    ].head(30)
)


Potential governance exceptions: 24,121


,User_ID,Role,CRM_Action,Device_Type,Data_Sensitivity,Failed_Logins,Anomaly_Score,Policy_Compliance_Score,Governance_Score,Permission_Granted,Baseline_Authorization,Contextual_Risk_Level,Access_Decision,Proposed_Access_Decision,Triggered_Rule_ID
0,1,Admin,ViewLead,Managed,5,1,0.114,76.190,51.810,True,ALLOW,MEDIUM,Block,REVIEW,CTX-004
1,2,Sales Rep,ViewLead,Managed,5,1,0.464,62.870,38.400,True,ALLOW,HIGH,Block,REVIEW,CTX-003
3,4,Analyst,ViewLead,Managed,3,0,0.181,85.890,57.530,True,ALLOW,LOW,Block,ALLOW,DEFAULT-001
5,6,Analyst,ViewLead,BYOD,5,1,0.207,75.770,49.330,True,ALLOW,MEDIUM,Block,REVIEW,CTX-004
9,10,Manager,EditLead,BYOD,5,0,0.296,73.980,48.480,True,ALLOW,MEDIUM,Block,REVIEW,CTX-005
10,11,Sales Rep,ViewLead,BYOD,4,1,0.137,89.910,57.030,True,ALLOW,HIGH,Block,REVIEW,CTX-003
12,13,Sales Rep,ViewLead,Managed,5,2,0.378,92.440,48.910,True,ALLOW,MEDIUM,Block,REVIEW,CTX-004
14,15,Manager,CreateOpportunity,Managed,4,0,0.433,92.490,52.540,True,ALLOW,MEDIUM,Block,REVIEW,CTX-005
15,16,Manager,EditLead,Managed,5,0,0.154,89.870,57.620,True,ALLOW,LOW,Block,ALLOW,DEFAULT-001
18,19,Manager,CreateOpportunity,Managed,2,1,0.352,91.540,54.250,True,ALLOW,MEDIUM,Block,REVIEW,CTX-005


# 19. Rule-Level Validation Summary

This table compares the observed historical synthetic block rate inside each proposed rule-triggered group.

It helps identify:

- rules strongly aligned with prior evidence;
- rules that may be overly strict;
- rules that may be too permissive;
- rules requiring additional business validation.


In [18]:
rule_validation = (
    df_rules.groupby("Triggered_Rule_ID")
    .agg(
        Records=("User_ID", "size"),
        Original_Block_Rate=("Is_Blocked", "mean")
    )
    .reset_index()
)

rule_validation["Original_Block_Rate"] *= 100

rule_validation = rule_validation.merge(
    rule_catalog[
        [
            "Rule_ID",
            "Category",
            "Proposed_Decision",
            "Rationale"
        ]
    ],
    left_on="Triggered_Rule_ID",
    right_on="Rule_ID",
    how="left"
)

display(
    rule_validation
    .sort_values("Original_Block_Rate", ascending=False)
    .round(2)
)


,Triggered_Rule_ID,Records,Original_Block_Rate,Rule_ID,Category,Proposed_Decision,Rationale
4,CTX-002,428,100.000,CTX-002,Contextual Risk,BLOCK,Highly sensitive access from BYOD with anomalo...
3,CTX-001,16,100.000,CTX-001,Contextual Risk,BLOCK,Multiple concurrent contextual risk signals in...
0,AUTH-001,15931,99.990,AUTH-001,Authorization,BLOCK,No explicit permission is available for the re...
5,CTX-003,986,99.590,CTX-003,Contextual Risk,REVIEW,High contextual risk requires manual or second...
6,CTX-004,4044,91.690,CTX-004,Contextual Risk,REVIEW,Sensitive-data access combined with authentica...
7,CTX-005,9745,89.490,CTX-005,Contextual Risk,REVIEW,Moderate contextual risk requires additional a...
1,AUTH-002,2968,76.620,AUTH-002,Authorization,BLOCK,Requested action is outside the proposed role ...
2,AUTH-003,812,75.250,AUTH-003,Authorization,REVIEW,Action requires additional approval or context...
8,DEFAULT-001,15070,62.400,DEFAULT-001,Default,ALLOW,Authorized low-risk access follows the default...


# 20. Rule Traceability Matrix

A governance rule should be traceable from evidence to policy, future control, and monitoring.

This matrix becomes a reusable artifact for later stages, especially:

- Stage 4 — Data Quality Framework;
- Stage 7 — Data Lineage & Traceability;
- Stage 8 — Stewardship & Governance Operating Model;
- Stage 10 — Governance Monitoring / Power BI.


In [19]:
rule_traceability = rule_catalog[
    [
        "Rule_ID",
        "Category",
        "Priority",
        "Condition",
        "Proposed_Decision",
        "Evidence_Source",
        "Policy_Status",
        "Control_Type"
    ]
].copy()

monitoring_map = {
    "AUTH-001": "Unauthorized-request rate",
    "AUTH-002": "RBAC baseline-block rate",
    "AUTH-003": "RBAC review rate",
    "CTX-001": "Critical contextual-risk rate",
    "CTX-002": "Sensitive BYOD anomaly rate",
    "CTX-003": "High contextual-risk review rate",
    "CTX-004": "Sensitive + failed-login review rate",
    "CTX-005": "Medium contextual-risk review rate",
    "DEFAULT-001": "Low-risk allow rate"
}

rule_traceability["Future_Monitoring_Metric"] = (
    rule_traceability["Rule_ID"].map(monitoring_map)
)

rule_traceability["Owner_Role"] = "TBD — Stage 8"
rule_traceability["Implementation_Status"] = "Prototype only"

display(rule_traceability)


,Rule_ID,Category,Priority,Condition,Proposed_Decision,Evidence_Source,Policy_Status,Control_Type,Future_Monitoring_Metric,Owner_Role,Implementation_Status
0,AUTH-001,Authorization,1,Permission_Granted = False,BLOCK,Stage 2 — permission analysis,Proposed,Preventive,Unauthorized-request rate,TBD — Stage 8,Prototype only
1,AUTH-002,Authorization,2,Role × CRM_Action baseline = BLOCK,BLOCK,Stage 3 — RBAC governance design,Proposed,Preventive,RBAC baseline-block rate,TBD — Stage 8,Prototype only
2,CTX-001,Contextual Risk,3,Contextual_Risk_Level = CRITICAL,BLOCK,Stages 1–2 — combined risk-factor evidence,Proposed,Preventive,Critical contextual-risk rate,TBD — Stage 8,Prototype only
3,CTX-002,Contextual Risk,4,High sensitivity AND BYOD AND high anomaly,BLOCK,Stage 2 — interaction evidence + governance de...,Proposed,Preventive,Sensitive BYOD anomaly rate,TBD — Stage 8,Prototype only
4,AUTH-003,Authorization,5,Role × CRM_Action baseline = REVIEW,REVIEW,Stage 3 — RBAC governance design,Proposed,Detective / Approval,RBAC review rate,TBD — Stage 8,Prototype only
5,CTX-003,Contextual Risk,6,Contextual_Risk_Level = HIGH,REVIEW,Stages 1–2 — combined risk-factor evidence,Proposed,Detective / Approval,High contextual-risk review rate,TBD — Stage 8,Prototype only
6,CTX-004,Contextual Risk,7,Data_Sensitivity >= 4 AND Failed_Logins > 0,REVIEW,Stages 1–2 — sensitivity and failed-login evid...,Proposed,Detective / Approval,Sensitive + failed-login review rate,TBD — Stage 8,Prototype only
7,CTX-005,Contextual Risk,8,Contextual_Risk_Level = MEDIUM,REVIEW,Stages 1–2 — combined risk-factor evidence,Proposed,Detective / Approval,Medium contextual-risk review rate,TBD — Stage 8,Prototype only
8,DEFAULT-001,Default,99,No higher-priority rule triggered,ALLOW,Stage 3 — governance design,Proposed,Default path,Low-risk allow rate,TBD — Stage 8,Prototype only


# 21. Governance Policy Table

This table consolidates the proposed policy catalog into a documentation-ready structure.

It remains a **prototype policy artifact** until organizational approval exists.


In [20]:
governance_policy_table = rule_catalog[
    [
        "Rule_ID",
        "Category",
        "Priority",
        "Condition",
        "Proposed_Decision",
        "Rationale",
        "Evidence_Source",
        "Policy_Status",
        "Control_Type"
    ]
].copy()

display(governance_policy_table)


,Rule_ID,Category,Priority,Condition,Proposed_Decision,Rationale,Evidence_Source,Policy_Status,Control_Type
0,AUTH-001,Authorization,1,Permission_Granted = False,BLOCK,No explicit permission is available for the re...,Stage 2 — permission analysis,Proposed,Preventive
1,AUTH-002,Authorization,2,Role × CRM_Action baseline = BLOCK,BLOCK,Requested action is outside the proposed role ...,Stage 3 — RBAC governance design,Proposed,Preventive
2,CTX-001,Contextual Risk,3,Contextual_Risk_Level = CRITICAL,BLOCK,Multiple concurrent contextual risk signals in...,Stages 1–2 — combined risk-factor evidence,Proposed,Preventive
3,CTX-002,Contextual Risk,4,High sensitivity AND BYOD AND high anomaly,BLOCK,Highly sensitive access from BYOD with anomalo...,Stage 2 — interaction evidence + governance de...,Proposed,Preventive
4,AUTH-003,Authorization,5,Role × CRM_Action baseline = REVIEW,REVIEW,Action requires additional approval or context...,Stage 3 — RBAC governance design,Proposed,Detective / Approval
5,CTX-003,Contextual Risk,6,Contextual_Risk_Level = HIGH,REVIEW,High contextual risk requires manual or second...,Stages 1–2 — combined risk-factor evidence,Proposed,Detective / Approval
6,CTX-004,Contextual Risk,7,Data_Sensitivity >= 4 AND Failed_Logins > 0,REVIEW,Sensitive-data access combined with authentica...,Stages 1–2 — sensitivity and failed-login evid...,Proposed,Detective / Approval
7,CTX-005,Contextual Risk,8,Contextual_Risk_Level = MEDIUM,REVIEW,Moderate contextual risk requires additional a...,Stages 1–2 — combined risk-factor evidence,Proposed,Detective / Approval
8,DEFAULT-001,Default,99,No higher-priority rule triggered,ALLOW,Authorized low-risk access follows the default...,Stage 3 — governance design,Proposed,Default path


# 22. Governance Control Architecture

```text
Access Request
      |
      v
Explicit Permission?
      |
  +---+---+
  |       |
 NO      YES
  |       |
BLOCK     v
      Role × Action Baseline
          |
      +---+---+---+
      |       |   |
    BLOCK   REVIEW ALLOW
      |       |     |
    BLOCK     |     v
              |  Contextual Risk
              |     |
              | +---+---+---+
              | |       |   |
              |CRITICAL HIGH/MED LOW
              | |       |   |
              |BLOCK   REVIEW ALLOW
              |
            REVIEW
```

### Governance principle

Authorization answers **“may this role perform this action?”**

Contextual risk answers **“is this access appropriate under the current conditions?”**

The two layers must remain conceptually separate.


# 23. Governance Rule Documentation and Lifecycle

Every production rule should eventually include:

- Rule ID
- Rule name
- Business owner
- Data / system owner
- Description
- Trigger condition
- Decision outcome
- Priority
- Evidence / justification
- Data fields used
- Exception process
- Monitoring metric
- Review frequency
- Effective date
- Last review date
- Change history
- Approval status
- Implementation status

### Lifecycle

`Draft → Review → Approved → Implemented → Monitored → Revised / Retired`

The current notebook represents only the **Draft / Prototype** stage.


# 24. Findings to Document

## 24.1 Authorization

- **Roles with the broadest proposed access:** `Admin` has the broadest proposed access, with all six CRM actions classified as `ALLOW`. `Manager` follows, with four actions classified as `ALLOW` and two (`ExportCRM` and `DeleteLead`) requiring `REVIEW`.

- **Most restricted CRM actions:** `DeleteLead` is the most restricted action in the proposed RBAC baseline. It is `ALLOW` only for Admin, requires `REVIEW` for Manager, and is `BLOCK` for Sales Rep, Support, and Analyst. `ExportCRM` and `ApproveDiscount` also receive elevated restrictions across multiple roles.

- **Role-action combinations requiring review:** the proposed matrix requires review for Manager–ExportCRM, Manager–DeleteLead, Sales Rep–ApproveDiscount, Sales Rep–ExportCRM, Support–EditLead, and Analyst–ExportCRM.

## 24.2 Contextual Risk

- **Most frequently triggered contextual factors:** To be completed after executing the reformulated contextual-risk model.

- **HIGH and CRITICAL risk frequency:** To be completed after executing the reformulated contextual-risk model.

- **Combinations deserving stricter review:** combinations involving high data sensitivity together with BYOD, anomalous behavior, failed login attempts, or multiple concurrent contextual signals should receive enhanced review. The proposed framework treats the combination of high sensitivity + BYOD + high anomaly as a candidate for blocking.

## 24.3 Rule Validation

- **Rules most strongly aligned with the original synthetic block pattern:** `AUTH-001` is expected to remain strongly aligned because the explanatory analysis showed that accesses without explicit permission were almost always blocked. Contextual BLOCK rules should be reassessed after execution of the reformulated risk model.

- **Potentially overly strict rules:** `AUTH-002` and any contextual rule producing `BLOCK` despite a substantial proportion of historical `Review` outcomes should be treated as candidates for reassessment.

- **Potentially too permissive rules:** the default `ALLOW` path should receive particular attention if a substantial proportion of records assigned to it were historically blocked.

## 24.4 Exceptions

- **Most common disagreement patterns:** expected disagreement patterns include historical `Block` records classified as proposed `REVIEW` or `ALLOW`, and historical `Review` records classified as proposed `BLOCK`.

- **Concentration by role or action:** this should be evaluated after execution by grouping disagreement records by `Role`, `CRM_Action`, and `Triggered_Rule_ID`.

- **Rules that should be reconsidered:** rules with high disagreement rates, especially BLOCK rules affecting historically mixed outcomes and the default ALLOW rule if it contains many historical blocks, should be prioritized for review.

## 24.5 Governance Conclusions

1. **Authorization and contextual risk should remain separate governance layers.** Role-based authorization defines what a user is normally permitted to do, while contextual signals determine whether an otherwise authorized access should be allowed, reviewed, or blocked under current conditions.

2. **Analytical evidence should inform governance policy without automatically becoming policy.** Thresholds, risk combinations, and access decisions in this notebook are proposed controls that require organizational validation before implementation.

3. **Governance rules require traceability, exception handling, monitoring, and periodic review.** A rule is not complete merely because it produces a decision; its rationale, evidence source, owner, monitoring metric, exceptions, and lifecycle must also be documented.

# 25. Limitations and Governance Boundaries

1. The `Role × CRM_Action` matrix is a **proposed governance design**, not an observed business policy.
2. The dataset is synthetic.
3. The dataset does not contain actual customer-level PII.
4. `ALLOW` does not exist in the original `Access_Decision` target.
5. Contextual thresholds are exploratory and data-driven.
6. The current risk score assigns equal weights for interpretability.
7. `Governance_Score` is deliberately excluded from operational rules because Stage 2 identified a proxy/leakage-like risk.
8. Statistical association does not automatically define policy.
9. A real implementation would require validation by business, security, IAM, privacy, legal/compliance, data owners, and system administrators.
10. Legal compliance cannot be inferred solely from these analytical rules.
11. Rules must be periodically reviewed for effectiveness, false positives, operational burden, and unintended access restrictions.


# 26. Stage 3 Deliverables

This notebook produces the following reusable governance artifacts:

- proposed internal sensitivity policy;
- proposed `Role × CRM_Action` RBAC matrix;
- transparent contextual-risk flags;
- contextual-risk score and levels;
- evidence-to-policy boundary table;
- prioritized governance rule catalog;
- deterministic decision engine;
- exception / disagreement dataset;
- rule-level validation summary;
- governance policy table;
- rule traceability matrix;
- governance rule lifecycle requirements;
- documented limitations and approval boundaries.

These outputs become inputs for later quality, metadata, privacy, lineage, stewardship, monitoring, and AI-governance stages.


# 27. Next Step — Stage 4 of 12: Data Quality Framework

The next project stage is:

## Stage 4 — Data Quality Framework

Stage 4 will convert governance expectations into measurable data-quality controls.

Planned outputs include:

- data-quality dimensions;
- rule IDs and severity levels;
- completeness rules;
- validity and domain rules;
- uniqueness checks;
- consistency checks;
- quality thresholds;
- pass/fail logic;
- Data Quality Score;
- issue register;
- linkage between governance rules and data-quality controls.

### Handoff from Stage 3

Stage 4 should reuse relevant Stage 3 artifacts, especially:

- approved domains and expected values;
- role/action governance mappings;
- required authorization fields;
- contextual-risk input fields;
- rule IDs and traceability metadata.

The objective is to make the governance framework **measurable, testable, and monitorable**.
